In [ ]:
"""
For Google Colab.
"""

# Clone repo
%cd /content
!git clone https://github.com/AHHHHHH0-0/Extending-CoFinDiff.git
%cd Extending-CoFinDiff

# Open terminal and swith branch 

## Import


In [ ]:
import sys
import yaml
import torch
import wandb
import numpy as np
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from pathlib import Path
from tqdm import tqdm

sys.path.insert(0, str(Path().resolve().parent.parent))
from denoiser.unet_model_ca_film import UNetDenoiserCAFilm  
from preprocessing.condition_encoder import MicroConditionEncoder
from diffusion.diffusion_ca_film import DiffusionCAFilm 
from training import train_step, validate, FinancialDataset
from config import training_config, project_config, denoiser_config, diffusion_config

## Config


In [ ]:
# Set seed 
SEED = project_config.SEED
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Device
DEVICE = project_config.DEVICE

# Fixed hyperparameters (not swept)
BS = training_config.BATCH_SIZE
EP = training_config.EPOCHS
WD = training_config.WEIGHT_DECAY
PU = training_config.P_UNCOND
ES = training_config.EARLY_STOPPING

print(f"Device: {DEVICE}")
print(f"Fixed hyperparameters:")
print(f"  Batch size: {BS}")
print(f"  Max epochs: {EP}")
print(f"  Weight decay: {WD}")
print(f"  P(uncond): {PU}")
print(f"  Early stopping: {ES}")
print(f"\nSwept: learning_rate, base_channels, num_res_blocks, beta_schedule")

## Load data


In [ ]:
# Load dataset
dataset = FinancialDataset('../../data/train/train_data_2d.json')

# Inspect a sample
sample = dataset[0]
print(f"\nSample structure:")
for key, val in sample.items():
    print(f"  {key}: shape {val.shape}, dtype {val.dtype}")

In [ ]:
# 80-20 train-val split
train_indices, val_indices = train_test_split(
    range(len(dataset)),
    test_size=0.2,
    random_state=SEED,
    shuffle=True
)

print(f"Train-Validation Split Setup:")
print(f"  Total samples: {len(dataset)}")
print(f"  Train samples: {len(train_indices)} ({len(train_indices)/len(dataset)*100:.1f}%)")
print(f"  Validation samples: {len(val_indices)} ({len(val_indices)/len(dataset)*100:.1f}%)")

## Sweep set-up


In [ ]:
# Load sweep config from YAML
with open("../../sweep_config.yaml", "r") as f:
    sweep_config = yaml.safe_load(f)

print("Sweep config:")
print(f"- Method: {sweep_config['method']}")
print(f"- Metric: {sweep_config['metric']['name']} ({sweep_config['metric']['goal']})")
print(f"- Parameters:")
for name, spec in sweep_config['parameters'].items():
    print(f"    {name}: {spec}")
print(f"- Early termination: {sweep_config['early_terminate']}")

In [ ]:
# Create the sweep
sweep_id = wandb.sweep(
    sweep_config,
    entity="ahliu7-uci",
    project="Extending-CoFinDiff",
)

print(f"Sweep ID: {sweep_id}")

## Training function


In [ ]:
def train_sweep():
    """Single training run called by the wandb sweep agent."""
    run = wandb.init()
    cfg = wandb.config

    # Sweep parameters
    LR = cfg.learning_rate
    BASE_CH = cfg.base_channels
    NUM_RES = cfg.num_res_blocks
    BETA_SCHED = cfg.beta_schedule

    # Log full config for reproducibility
    wandb.config.update({
        # Training config (fixed)
        "batch_size": BS,
        "epochs": EP,
        "weight_decay": WD,
        "p_uncond": PU,
        "early_stopping": ES,
        # Denoiser config (fixed)
        "down_sample_kernel_size": denoiser_config.DOWN_SAMPLE_KERNEL_SIZE,
        "down_sample_stride": denoiser_config.DOWN_SAMPLE_STRIDE,
        "down_sample_padding": denoiser_config.DOWN_SAMPLE_PADDING,
        "up_sample_kernel_size": denoiser_config.UP_SAMPLE_KERNEL_SIZE,
        "up_sample_padding": denoiser_config.UP_SAMPLE_PADDING,
        "num_res_blocks": NUM_RES,  # swept
        "res_block_kernel_size": denoiser_config.RES_BLOCK_KERNEL_SIZE,
        "res_block_num_groups": denoiser_config.RES_BLOCK_NUM_GROUPS,
        "res_block_dropout": denoiser_config.RES_BLOCK_DROPOUT,
        "cross_attn_num_heads": denoiser_config.CROSS_ATTN_NUM_HEADS,
        "cross_attn_scale": denoiser_config.CROSS_ATTN_SCALE,
        "cond_context_dim": denoiser_config.COND_CONTEXT_DIM,
        "film_hidden_dim": denoiser_config.FILM_HIDDEN_DIM,
        "num_macro_scalars": denoiser_config.NUM_MACRO_SCALARS,
        "time_embed_dim": denoiser_config.TIME_EMBED_DIM,
        "channel_mult": denoiser_config.CHANNEL_MULT,
        # Diffusion config (fixed)
        "timesteps": diffusion_config.TIMESTEPS,
        "objective": diffusion_config.OBJECTIVE,
        "beta_schedule": BETA_SCHED,  # swept
        "beta_start": diffusion_config.BETA_START,
        "beta_end": diffusion_config.BETA_END,
        "auto_normalize": diffusion_config.AUTO_NORMALIZE,
        "guidance_scale": diffusion_config.GUIDANCE_SCALE,
        "return_trajectory": diffusion_config.RETURN_TRAJECTORY,
    }, allow_val_change=True)
    print("WandB config logged")

    # Data loaders
    train_loader = DataLoader(
        dataset.get_subset(train_indices),
        batch_size=BS,
        shuffle=True,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False,
        generator=torch.Generator().manual_seed(SEED)
    )
    val_loader = DataLoader(
        dataset.get_subset(val_indices),
        batch_size=BS,
        shuffle=False,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    print("Data loaders created")

    # Models — base_channels from sweep
    denoiser = UNetDenoiserCAFilm(in_channels=1, base_channels=BASE_CH, num_res_blocks=NUM_RES).to(DEVICE)
    micro_encoder = MicroConditionEncoder().to(DEVICE)
    diffusion = DiffusionCAFilm(beta_schedule=BETA_SCHED)
    param_count = denoiser.get_num_parameters()
    wandb.config.update({"denoiser_params": param_count}, allow_val_change=True)
    print("Models created")

    # Optimizer — learning_rate from sweep
    optimizer = torch.optim.Adam(
        list(denoiser.parameters()) + list(micro_encoder.parameters()),
        lr=LR,
        weight_decay=WD
    )
    print("Optimizer created")
    # LR Scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EP)
    print("LR Scheduler created")

    # Training tracking
    best_val_loss = float('inf')
    epochs_without_improvement = 0

    # Training Loop
    for epoch in range(EP):
        denoiser.train()
        micro_encoder.train()

        epoch_train_loss = 0.0
        num_train_batches = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EP}", leave=False)

        for batch in pbar:
            x0 = batch['returns_2d'].unsqueeze(1).to(DEVICE)

            conditions = {
                'trend': batch['trend'].to(DEVICE),
                'realized_vol': batch['realized_vol'].to(DEVICE),
                'interest_rate': batch['interest_rate'].to(DEVICE),
                'volatility_index': batch['volatility_index'].to(DEVICE)
            }

            loss = train_step(
                denoiser=denoiser,
                diffusion=diffusion,
                x=x0,
                conditions=conditions,
                micro_encoder=micro_encoder,
                optimizer=optimizer,
                p_uncond=PU
            )

            epoch_train_loss += loss
            num_train_batches += 1
            pbar.set_postfix({"loss": f"{loss:.6f}"})

        avg_train_loss = epoch_train_loss / num_train_batches
        avg_val_loss = validate(denoiser, micro_encoder, diffusion, val_loader, DEVICE)

        wandb.log({
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
        })

        scheduler.step()

        if (epoch + 1) % 50 == 0 or epoch == 0:
            print(f"Epoch {epoch + 1}/{EP} - Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}")

        # Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= ES:
            print(f"\nEarly stopping at epoch {epoch + 1}")
            print(f"Best validation loss: {best_val_loss:.6f}")
            break

    # Log best val loss
    wandb.run.summary["best_val_loss"] = best_val_loss

    print(f"\nRun complete — best val_loss: {best_val_loss:.6f}")
    run.finish()

## Run sweep


In [ ]:
# Run the sweep agent — adjust count for number of runs
SWEEP_COUNT = 30

wandb.agent(sweep_id, function=train_sweep, count=SWEEP_COUNT)